In [1]:
import jax
import jax.numpy as jnp
from jax.scipy.linalg import expm
import numpyro
import numpyro.distributions as dist
from numpyro.infer import MCMC, NUTS


def lou_model(N, Nsub, K, R, p, ID, repme, cumu, deltat, X, Y, missing_ID, 
              ncate4, ncate5, ncate6, ncate7):
    """
    NumPyro implementation of the Latent Ornstein-Uhlenbeck model.
    Based on Tran et al. (2021) and the provided Stan code.
    """
    
    # --- 1. Priors for Item Parameters ---
    mu_theta = numpyro.sample("mu_theta", dist.Normal(0., 10.))
    sigma_theta = numpyro.sample("sigma_theta", dist.HalfCauchy(5.))
    
    # Binary item thresholds (items 1-3)
    theta1 = numpyro.sample("theta1", dist.Normal(mu_theta, sigma_theta))
    theta2 = numpyro.sample("theta2", dist.Normal(mu_theta, sigma_theta))
    theta3 = numpyro.sample("theta3", dist.Normal(mu_theta, sigma_theta))
    
    # Ordinal item thresholds (items 4-7)
    theta4 = numpyro.sample("theta4", dist.TransformedDistribution(
        dist.Normal(mu_theta, sigma_theta).expand([ncate4 - 1]), 
        dist.transforms.OrderedTransform()))
    theta5 = numpyro.sample("theta5", dist.TransformedDistribution(
        dist.Normal(mu_theta, sigma_theta).expand([ncate5 - 1]), 
        dist.transforms.OrderedTransform()))
    theta6 = numpyro.sample("theta6", dist.TransformedDistribution(
        dist.Normal(mu_theta, sigma_theta).expand([ncate6 - 1]), 
        dist.transforms.OrderedTransform()))
    theta7 = numpyro.sample("theta7", dist.TransformedDistribution(
        dist.Normal(mu_theta, sigma_theta).expand([ncate7 - 1]), 
        dist.transforms.OrderedTransform()))

    # Discrimination parameters (lambda)
    sigma_lambda = numpyro.sample("sigma_lambda", dist.HalfCauchy(5.))
    lambdas = numpyro.sample("lambda", dist.Normal(jnp.ones(K), sigma_lambda))
    
    # Regression coefficients and Random Effects
    beta = numpyro.sample("beta", dist.Cauchy(jnp.zeros((K, p)), 5.))
    sigma_bk = numpyro.sample("sigma_bk", dist.HalfCauchy(jnp.ones(K) * 5.))
    b_raw = numpyro.sample("b_raw", dist.Normal(jnp.zeros((Nsub, K)), 1.))
    b = b_raw * sigma_bk  # Non-centered parameterization 

    # --- 2. OU Process Parameters (Gamma and Omega) ---
    gamma_flat = numpyro.sample("gamma_flat", dist.Normal(jnp.zeros(R * R), 10.))
    Gamma = gamma_flat.reshape((R, R))
    
    # Constraints for Complex/Real eigenvalues (Model 2a) 
    constraint1 = Gamma[0, 0] + Gamma[1, 1]
    constraint2 = Gamma[0, 0] * Gamma[1, 1] - Gamma[0, 1] * Gamma[1, 0]
    numpyro.factor("stability_constraint", 
                   jnp.where((constraint1 > 0) & (constraint2 > 0), 0., -jnp.inf))

    rho = numpyro.sample("rho", dist.Uniform(-1., 1.))
    # Correlation matrix Omega [cite: 740]
    Omega = jnp.array([[1., rho], [rho, 1.]])

    # --- 3. Latent Variables (xi) Evolution ---
    xi = jnp.zeros((N, R))
    
    # Evolution loop per subject
    for i in range(Nsub):
        start_idx = cumu[i] - repme[i]
        
        # Initial state at time = 1 [cite: 746]
        xi_first = numpyro.sample(f"xi_{start_idx}", dist.MultivariateNormal(jnp.zeros(R), Omega))
        xi = xi.at[start_idx].set(xi_first)
        
        # Subsequent states [cite: 748, 749]
        for j in range(1, repme[i]):
            k = start_idx + j
            dt = deltat[k]
            
            # Matrix exponential for transition
            ExpGamma = expm(-dt * Gamma)
            mean_xi = jnp.matmul(ExpGamma, xi[k-1])
            
            # Conditional covariance
            Cova_trans = Omega - jnp.matmul(ExpGamma, jnp.matmul(Omega, ExpGamma.T))
            
            xi_next = numpyro.sample(f"xi_{k}", dist.MultivariateNormal(mean_xi, Cova_trans))
            xi = xi.at[k].set(xi_next)

    # --- 4. Likelihood ---
    for i in range(N):
        sub_id = ID[i] - 1  # Adjust for 0-indexing
        
        # Binary Items (1-3)
        for k in range(3):
            if missing_ID[i, k] == 0:
                theta_bin = [theta1, theta2, theta3][k]
                logits = theta_bin + jnp.dot(beta[k], X[i]) + lambdas[k] * xi[i, 0] + b[sub_id, k]
                numpyro.sample(f"Y_{i}_{k}", dist.Bernoulli(logits=logits), obs=Y[i, k])
        
        # Ordinal Items (4-7) [cite: 750]
        ordinal_items = [(3, theta4), (4, theta5), (5, theta6), (6, theta7)]
        for k, th in ordinal_items:
            if missing_ID[i, k] == 0:
                eta = jnp.dot(beta[k], X[i]) + lambdas[k] * xi[i, 1] + b[sub_id, k]
                # Ordered Logistic likelihood
                numpyro.sample(f"Y_{i}_{k}", dist.OrderedLogistic(eta, th), obs=Y[i, k])

/u/zwu1/.conda/envs/ou/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
import numpy as np

def simulate_study(shifted_mean=False, Nsub=600, p=2, q=2):
    """
    Simulates longitudinal IRT data.
    Binary items (1-3) -> 0/1 for bernoulli_logit.
    Ordinal items (4-7) -> 1+ for ordered_logistic.
    """
    K, R = 7, 2
    repme = np.random.randint(2, 13, size=Nsub)
    N = np.sum(repme)
    
    ID = np.repeat(np.arange(1, Nsub + 1), repme)
    cumu = np.cumsum(repme)
    
    # Covariates
    X = np.random.normal(size=(N, p))
    Z = np.random.binomial(1, 0.5, size=(N, q)) 
    
    # --- 1. Latent Process Parameters ---
    Gamma = np.array([[0.18, -0.07], 
                      [0.10, 0.15]])
    rho = 0.60
    Omega = np.array([[1.0, rho], [rho, 1.0]])
    
    gamma_latent = np.zeros((R, q))
    if shifted_mean:
        gamma_latent = np.array([[0.8, 1.2], [0.5, 0.9]])

    xi = np.zeros((N, R))
    deltat = np.zeros(N)
    time = np.zeros(N)
    
    for i in range(Nsub):
        start = cumu[i] - repme[i]
        t_subj = np.concatenate(([0], np.cumsum(np.random.uniform(0.5, 1.5, size=repme[i]-1))))
        time[start : start + repme[i]] = t_subj
        
        mu_start = (gamma_latent @ Z[start]) * time[start]
        xi[start] = np.random.multivariate_normal(mu_start, Omega)
        
        for j in range(1, repme[i]):
            k = start + j
            dt = t_subj[j] - t_subj[j-1]
            deltat[k] = dt
            Phi = expm(-dt * Gamma)
            Q = Omega - Phi @ Omega @ Phi.T
            
            target_k = (gamma_latent @ Z[k]) * time[k]
            target_prev = (gamma_latent @ Z[k-1]) * time[k-1]
            cond_mean = target_k + Phi @ (xi[k-1] - target_prev)
            # Add jitter in simulation for parity with Stan stability
            xi[k] = np.random.multivariate_normal(cond_mean, Q + np.eye(R)*1e-9)

    # --- 2. Measurement Model Parameters ---
    lam = np.zeros(K)
    lam[0], lam[1], lam[2] = 1.20, 4.00, 4.10  
    lam[3], lam[4], lam[5], lam[6] = 3.10, 5.20, 3.00, 1.70 

    B = np.zeros((K, p))
    B[1, :] = [0.10, 0.20]   
    B[4, :] = [0.30, -0.30]  

    sig_b = np.zeros(K)
    sig_b[0], sig_b[2], sig_b[3], sig_b[6] = 3.70, 4.80, 3.10, 1.70
    b = np.random.normal(0, 1, size=(Nsub, K)) * sig_b
    
    Y = np.zeros((N, K), dtype=int)
    def inv_logit(x): return 1 / (1 + np.exp(-x))

    for i in range(N):
        sub_idx = ID[i] - 1
        for k in range(K):
            f_idx = 0 if k < 3 else 1
            eta = X[i] @ B[k] + lam[k] * xi[i, f_idx] + b[sub_idx, k]

            # BINARY ITEMS (1, 2, 3) -> Must be 0 or 1
            if k < 3:
                # Map theta values from table for binary intercepts if provided
                # Else use 0.0 or a custom baseline
                theta_base = 0.0 
                if k == 0: theta_base = 2.30 # Using first theta1 from table
                
                prob = inv_logit(theta_base + eta)
                Y[i, k] = np.random.binomial(1, prob)

            # ORDINAL ITEMS (4, 5, 6, 7) -> Must be 1, 2, 3...
            else:
                if k == 4: # Item 5 (theta_51, 52, 53)
                    th = [-7.50, -2.50, 2.60]
                elif k == 6: # Item 7 (theta_71, 72, 73)
                    th = [-4.30, -1.00, 1.40]
                else: # Items 4 and 6 (binary logic but in ordered_logistic)
                    th = [0.0]

                p_cum = [inv_logit(t - eta) for t in th]
                probs = np.diff([0] + p_cum + [1])
                # Shift to 1-based indexing for ordered_logistic
                Y[i, k] = np.random.choice(np.arange(1, len(probs) + 1), 
                                           p=np.clip(probs, 0, 1) / np.sum(probs))

    return {
        'N': N, 'Nsub': Nsub, 'K': K, 'R': R, 'p': p, 'q': q,
        'ID': ID, 'cumu': cumu, 'repme': repme, 'Y': Y,
        'missing_ID': np.zeros((N, K), dtype=int),
        'deltat': deltat, 'time': time, 'X': X, 'Z': Z,
        'ncate4': 2, 'ncate5': 4, 'ncate6': 2, 'ncate7': 4, 
        'true_xi': xi, 'true_gamma': gamma_latent, 
        'true_lambda': lam, 'true_Gamma_mat': Gamma
    }

In [4]:
import jax
import numpy as np
import jax.numpy as jnp
from numpyro.infer import MCMC, NUTS
from scipy.linalg import expm as s_expm

# Ensure the lou_model function is imported or defined in the namespace
# from your_module import lou_model

def test_lou_model_execution():
    # 1. Generate Synthetic Data
    # Nsub=600, K=7, R=2 as used in the study [cite: 179, 180]
    print("Generating simulation data...")
    sim_data = simulate_study(shifted_mean=False, Nsub=50, p=2, q=2) 
    # Note: Nsub reduced to 50 for a faster integration test

    # 2. Prepare Data for NumPyro
    # NumPyro requires JAX arrays and specific data types [cite: 733, 735]
    numpyro_data = {
        "N": sim_data['N'],
        "Nsub": sim_data['Nsub'],
        "K": sim_data['K'],
        "R": sim_data['R'],
        "p": sim_data['p'],
        "ID": jnp.array(sim_data['ID']),
        "repme": jnp.array(sim_data['repme']),
        "cumu": jnp.array(sim_data['cumu']),
        "deltat": jnp.array(sim_data['deltat']),
        "X": jnp.array(sim_data['X']),
        "Y": jnp.array(sim_data['Y']),
        "missing_ID": jnp.array(sim_data['missing_ID']),
        "ncate4": sim_data['ncate4'],
        "ncate5": sim_data['ncate5'],
        "ncate6": sim_data['ncate6'],
        "ncate7": sim_data['ncate7']
    }

    # 3. Setup Inference (MCMC/NUTS)
    # Using NUTS kernel as it is the standard for Stan-like models [cite: 233, 651]
    nuts_kernel = NUTS(lou_model)
    mcmc = MCMC(
        nuts_kernel, 
        num_warmup=100, 
        num_samples=100, 
        num_chains=1
    )

    # 4. Run Model
    print("Starting MCMC test run...")
    rng_key = jax.random.PRNGKey(0)
    try:
        mcmc.run(rng_key, **numpyro_data)
        print("MCMC test run completed successfully.")
        
        # 5. Basic Verification of Posterior Samples
        samples = mcmc.get_samples()
        
        # Verify specific parameters from the paper are present [cite: 202, 215]
        expected_params = ["Gamma", "rho", "lambda", "mu_theta", "sigma_theta"]
        for param in expected_params:
            assert param in samples, f"Parameter {param} missing from posterior samples."
        
        # Check stability constraints for Gamma [cite: 164, 229]
        gamma_samples = samples["Gamma"]
        mean_gamma = jnp.mean(gamma_samples, axis=0)
        c1 = mean_gamma[0, 0] + mean_gamma[1, 1]
        c2 = mean_gamma[0, 0] * mean_gamma[1, 1] - mean_gamma[0, 1] * mean_gamma[1, 0]
        
        print(f"Mean Gamma Stability: Constraint1={c1:.4f}, Constraint2={c2:.4f}")
        assert c1 > 0 and c2 > 0, "Posterior Gamma violates stability constraints."

    except Exception as e:
        print(f"Test failed with error: {e}")

if __name__ == "__main__":
    test_lou_model_execution()

Generating simulation data...


/tmp/ipykernel_1267974/866199837.py:53: RuntimeWarning: covariance is not symmetric positive-semidefinite.
  xi[k] = np.random.multivariate_normal(cond_mean, Q + np.eye(R)*1e-9)


Starting MCMC test run...
Test failed with error: 
